# Aşama 2 — güçlü artırma, yüksek çözünürlük, dengeli örnekleme, TTA (Colab)

Çalışma zamanı: **T4 GPU**. Kurallar `docs/protokol_asama2.md` dosyasındadır.

Her model için tek koşu: `--aug guclu --image-size 384 --balanced --amp`. TTA yeniden eğitim istemez; aynı ağırlıkla ölçülür.

**Seçim yalnız doğrulama macro-F1 ile yapılır.** Test her yapılandırma için bir kez ölçülür ve karar için kullanılmaz.

Her model hücresinin sonunda o modelin sonuçları zip olarak iner. Oturum koparsa: kurulum hücrelerini çalıştır, son indirdiğin zip'i "Kaldığın yerden devam" hücresiyle yükle, bitmemiş modelden devam et.

In [ ]:
import torch
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'GPU kapalı. Çalışma zamanını T4 yapıp yeniden başlat.'

In [ ]:
%cd /content
!rm -rf Final-project
!git clone https://github.com/bernademirer/Final-project.git
%cd /content/Final-project
!git log --oneline -3
from pathlib import Path
assert 'guclu' in Path('src/train.py').read_text(encoding='utf-8'), "GitHub'daki kod eski: Aşama 2 değişiklikleri henüz push edilmemiş."

In [ ]:
!pip -q install -r requirements.txt datasets

In [ ]:
import pandas as pd
from datasets import load_dataset

splits = pd.read_csv('data/splits.csv')
assert len(splits) == 1457
assert (splits.kume == 'test').sum() == 292
assert splits.file_name.nunique() == 1457

out = Path('data/images')
out.mkdir(parents=True, exist_ok=True)
need = set(splits.file_name) - {p.name for p in out.glob('*.jpg')}
if need:
    for row in load_dataset('Layered-Labs/ACNE04')['train']:
        if row['file_name'] in need:
            row['image'].save(out / row['file_name'])
print('jpg', len(list(out.glob('*.jpg'))))
assert len(list(out.glob('*.jpg'))) == 1457

### Kaldığın yerden devam (yalnız oturum koptuysa)

Son indirdiğin `asama2_<model>.zip` dosyasını yükle. İçindeki `results.csv` önceki modellerin satırlarını da taşır.

In [ ]:
from google.colab import files
import zipfile
up = files.upload()
for name in up:
    zipfile.ZipFile(name).extractall('/content/Final-project')
!ls results

## Yardımcı fonksiyon

`AGIRLIK_INDIR = True` yaparsan `.pt` dosyası da zip'e girer (ResNet50 için ~95 MB). Ağırlık olmadan da tüm sayılar ve figürler iner.

In [ ]:
import json
import re
import subprocess
import zipfile
from pathlib import Path
import pandas as pd
from google.colab import files

MODELLER = ['resnet50', 'efficientnet_b0', 'convnext_tiny', 'densenet121']
AGIRLIK_INDIR = False


def calistir(cmd):
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    if p.wait():
        raise RuntimeError('komut hata verdi: ' + cmd)
    return ''.join(lines)


def macro_f1(cikti):
    return float(re.search(r'^macro_f1 ([0-9.]+)', cikti, re.M).group(1))


def csv_satirlari():
    yol = Path('results/results.csv')
    return set(pd.read_csv(yol).model) if yol.exists() else set()


def kos(model):
    ad = f'{model}_asama2'
    agirlik = f'results/{ad}.pt'
    val_eval = Path(f'results/{ad}_val_eval.json')
    if val_eval.exists() and f'{ad}_tta' in csv_satirlari():
        print(ad, 'zaten tamam, atlandı')
        return

    if not Path(agirlik).exists():
        calistir(f'python src/train.py --model {model} --aug guclu --image-size 384 --balanced --amp --name {ad}')

    # TTA karari icin dogrulama, ikisi de ayni kosulda (fp32) olculur
    v = macro_f1(calistir(f'python src/evaluate.py --model {model} --weights {agirlik} --name {ad} --split val'))
    v_tta = macro_f1(calistir(f'python src/evaluate.py --model {model} --weights {agirlik} --name {ad} --split val --tta'))
    val_eval.write_text(json.dumps({'val_macro_f1': v, 'val_macro_f1_tta': v_tta}, indent=2), encoding='utf-8')

    calistir(f'python src/evaluate.py --model {model} --weights {agirlik} --name {ad} --split test --write-csv')
    calistir(f'python src/evaluate.py --model {model} --weights {agirlik} --name {ad} --split test --tta --write-csv')

    paket = [f'results/{ad}_val.json', str(val_eval), f'results/{ad}_egri.png',
             f'results/cm_{ad}.png', f'results/cm_{ad}_tta.png', 'results/results.csv']
    paket += [str(p) for p in Path('results').glob('*_asama2_val_eval.json')]
    paket += [str(p) for p in Path('results').glob('*_asama2_val.json')]
    if AGIRLIK_INDIR:
        paket.append(agirlik)
    zip_yolu = f'/content/asama2_{model}.zip'
    with zipfile.ZipFile(zip_yolu, 'w') as z:
        for p in sorted(set(paket)):
            if Path(p).exists():
                z.write(p)
    print('val', v, 'val_tta', v_tta)
    files.download(zip_yolu)

## Modeller

Her hücre: eğitim → doğrulama (TTA'lı ve TTA'sız) → test (TTA'lı ve TTA'sız, `results.csv`'ye yazılır) → zip indirme. Model başına tahmini süre T4'te 20–60 dakika.

In [ ]:
kos('resnet50')

In [ ]:
kos('efficientnet_b0')

In [ ]:
kos('convnext_tiny')

In [ ]:
kos('densenet121')

## Özet ve seçim

Aşama 2 kazananı: doğrulama macro-F1'i en yüksek model; TTA, doğrulamada daha iyiyse eklenir. Test sütunları yalnız raporlama içindir.

In [ ]:
csv = pd.read_csv('results/results.csv').set_index('model')
satirlar = []
for m in MODELLER:
    ad = f'{m}_asama2'
    yol = Path(f'results/{ad}_val_eval.json')
    if not yol.exists():
        print('eksik:', m)
        continue
    val = json.loads(yol.read_text(encoding='utf-8'))
    satirlar.append({
        'model': m,
        'asama1_test_f1': csv.macro_f1.get(m),
        'val_f1': val['val_macro_f1'],
        'val_f1_tta': val['val_macro_f1_tta'],
        'test_f1': csv.macro_f1.get(ad),
        'test_f1_tta': csv.macro_f1.get(ad + '_tta'),
        'test_dogruluk': csv.dogruluk.get(ad),
        'epoch': csv.epoch.get(ad),
    })
ozet = pd.DataFrame(satirlar)
ozet['val_en_iyi'] = ozet[['val_f1', 'val_f1_tta']].max(axis=1)
secilen = ozet.loc[ozet.val_en_iyi.idxmax()]
tta = bool(secilen.val_f1_tta > secilen.val_f1)
kazanan = secilen.model + (' + TTA' if tta else '')
ozet.to_csv('results/asama2_ozet.csv', index=False)
Path('results/asama2_kazanan.txt').write_text(kazanan + '\n', encoding='utf-8')
print('Aşama 2 kazananı (doğrulamaya göre):', kazanan)
ozet.round(4)

In [ ]:
!python src/sekiller.py
paket = ['results/results.csv', 'results/asama2_ozet.csv', 'results/asama2_kazanan.txt', 'results/sekil_asama2.png']
paket += [str(p) for p in Path('results').glob('*asama2*') if p.suffix in ('.json', '.png')]
with zipfile.ZipFile('/content/asama2_sonuc.zip', 'w') as z:
    for p in sorted(set(paket)):
        if Path(p).exists():
            z.write(p)
files.download('/content/asama2_sonuc.zip')